# 01 — Requirements and payload

Survey geometry bounds camera groundspeed. Aircraft stall margins constrain
airspeed; wind must be reconciled in mission planning. Figures are engineering
predictions, not flight or survey accuracy measurements. Basler is preferred and
substitutable; the existing OAK/PX4 platform remains separately supported.


## Derivation and decision boundary

**Inputs:** `design_point.yaml: survey`, `mission/mission_params.yaml`,
`payload/survey_cage.yaml`. **Implementation:** `uavdesign/survey.py`,
`design.survey_speed_limit`, and `integration.resource_budget`.

For nadir viewing over level ground, with height $H$ [m], pixel pitch $p$ [m],
focal length $f$ [m], and image dimensions $N_x,N_y$ [pixels]:

$$g=Hp/f,\quad W=N_xg,\quad L=N_yg,\quad d=L(1-o_f)$$
$$f_{trigger}=V_g/d,\quad b_{px}=V_g t_e/g,\quad
V_{g,max}=\min(0.8 f_{cap}d,\;b_{max}g/t_e).$$

Plain-text equations: `GSD = height * pixel_pitch / focal_length`;
`trigger_spacing = image_length * (1 - forward_overlap)`;
`trigger_hz = groundspeed / trigger_spacing`; `blur_px = groundspeed * exposure / GSD`.

Here $g$ is GSD [m/pixel], $d$ trigger spacing [m], $o_f$ forward overlap,
$t_e$ exposure [s], and $V_g$ **groundspeed** [m/s]. The camera's maximum FPS
and the selected acquisition cap are distinct; both appear in the plot. The
0.8 frame-rate multiplier reserves timing headroom. Side overlap controls line
spacing $W(1-o_s)$, not forward triggering. Coverage uses that spacing, speed,
survey time and the configured on-line fraction; its upper bound does not include
transit or return allocations.

**External review:** establish the actual lens focus/distortion, exposure-event
latency and sustained image rate. GSD is sampling distance, not demonstrated
position accuracy. Terrain, oblique attitude, crosswind crab and rolling mission
altitude require a footprint model beyond this flat-ground calculation.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


In [ ]:
rows = []
for factor in studies["altitude_factors"]:
    height = s["agl_m"] * factor
    rows.append({"agl_m": height, "gsd_mm_px": cam.gsd_m(height) * 1000,
                 "swath_m": cam.swath_m(height), "along_track_m": cam.footprint_along_m(height),
                 "fps_speed_limit_m_s": sv.max_ground_speed_m_s(cam, height, s["forward_overlap"]),
                 "blur_speed_limit_m_s": sv.max_speed_for_blur(cam, height, s["exposure_s"], s["blur_limit_px"])})
display(pd.DataFrame(rows))
display(pd.DataFrame([p.summary() for p in d.payloads.values()]))


In [ ]:
heights = np.array(studies["altitude_factors"]) * s["agl_m"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(heights, [sv.max_ground_speed_m_s(cam, h, s["forward_overlap"]) for h in heights], label="Camera frame-rate limit (80% maximum)")
ax.plot(heights, [sv.max_ground_speed_m_s(acquisition_cam, h, s["forward_overlap"]) for h in heights], "--", label="Acquisition cap (80% of configured cap)")
for factor in studies["exposure_factors"]:
    exposure = s["exposure_s"] * factor
    ax.plot(heights, [sv.max_speed_for_blur(cam, h, exposure, s["blur_limit_px"]) for h in heights], label=f"Blur limit at {exposure*1e6:.0f} us")
ax.set(xlabel="Height above ground [m]", ylabel="Maximum ground speed [m/s]")
ax.legend(); ax.grid(alpha=.3)
save(fig, "01_survey_speed_limits.png")
rows = []
for name, ev in d.evaluations.items():
    coverage = sv.coverage_ha(cam, s["agl_m"], ev.cruise_speed_m_s, s["side_overlap"], ev.endurance_min*60, s["turn_efficiency"])
    rows.append({"config": name, "endurance_min": ev.endurance_min, "coverage_upper_bound_ha": coverage,
                 "survey_speed_compatible": ev.survey_speed_compatible})
display(pd.DataFrame(rows))
print("Coverage uses the full estimated endurance: deduct climb, transit, return and terrain allowances for a real plan.")
from uavdesign.integration import build_handoff
handoff = build_handoff()
display(pd.DataFrame([handoff["geometry"], handoff["resources"]]))
